將 ROOT_DIR 下的所有圖片先使用 yolo 偵測出花朵位置 (另存在 ROOT_DIR / "yolo_crops"
再將圖片以及 bbox 送給 SAM 去除背景 (另存在 ROOT_DIR / "sam_removed"
 

In [3]:
import os
import cv2
import torch
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from sam2.sam2_image_predictor import SAM2ImagePredictor

# ===========================
# 基本環境與參數設定
# ===========================
os.environ['CUDA_VISIBLE_DEVICES'] = '1,2,3'
print("Visible CUDA devices:", torch.cuda.device_count())

GPU_ID = 2
DEVICE_STR = f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu"

YOLO_WEIGHTS = "/home/nas2/Workspace/Aaron/DATA/ObjectDetect/flower_detect/FD11/weights/best.pt"
# ROOT_DIR     = Path("/home/nas2/Workspace/Aaron/phal_research/raw_images_0302")
ROOT_DIR     = Path("/home/nas2/Personal/Aaron/temp/")
IMG_SIZE     = 960
CONF_THRES   = 0.5
IOU_THRES    = 0.75
PAD_RATIO    = 0.20  # 策略三的 Padding 比例 (20%)

# ===========================
# 建立輸出資料夾
# ===========================
VIS_DIR = ROOT_DIR / "0_bbox_visual"
OUTPUT_DIR  = ROOT_DIR / "sam2_final_output" # 統一輸出最終結果

for d in [VIS_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

EXTS = {".jpg", ".JPG", ".jpeg", ".png", ".tif", ".tiff"}

# ===========================
# SAM2 Predictor 封裝
# ===========================
class SAM2Predictor:
    def __init__(self, device: str = "cuda:0"):
        self.predictor = SAM2ImagePredictor.from_pretrained("facebook/sam2.1-hiera-large")
        if hasattr(self.predictor, "to"): 
            self.predictor.to(device)
        self.device = device

    def set_image(self, image_bgr: np.ndarray):
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        self.predictor.set_image(image_rgb)

    def predict_with_boxes(self, boxes_xyxy: np.ndarray):
        masks_list = []
        for i in range(len(boxes_xyxy)):
            box = boxes_xyxy[i].astype(np.float32)
            masks, scores, _ = self.predictor.predict(
                point_coords=None,
                point_labels=None,
                box=box,
                multimask_output=False
            )
            m = masks[0].astype(bool)
            masks_list.append(m)
        return masks_list
        
# ===========================
# 初始化模型
# ===========================
print("[INFO] Loading YOLO model...")
yolo_model = YOLO(YOLO_WEIGHTS)
print("[INFO] Loading SAM2 model on", DEVICE_STR)
sam2 = SAM2Predictor(device=DEVICE_STR)
print("[INFO] Models loaded.\n")

# ===========================
# 輔助函數：套用 Mask 並裁切 (包含 OpenCV 破洞修復)
# ===========================
def apply_mask_and_crop(img_bgr, mask):
    if not mask.any():
        return None
        
    # --- OpenCV 填補內部破洞 (甜甜圈修復) ---
    mask_uint8 = (mask * 255).astype(np.uint8)
    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if contours:
        filled_mask = np.zeros_like(mask_uint8)
        cv2.drawContours(filled_mask, contours, -1, 255, thickness=-1)
        mask = filled_mask.astype(bool)
    # ---------------------------------------

    ys, xs = np.where(mask)
    y1, y2 = ys.min(), ys.max()
    x1, x2 = xs.min(), xs.max()
    
    # 將 BGR 轉換為帶有透明通道的 BGRA (4通道)
    img_bgra = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2BGRA)
    
    # 將 Mask 以外 (背景) 的區域，其 Alpha 通道設為 0 (完全透明)
    img_bgra[~mask, 3] = 0 
    
    # 根據 Bounding Box 裁切
    return img_bgra[y1:y2+1, x1:x2+1]

# ===========================
# 主流程
# ===========================
img_paths = sorted([p for p in ROOT_DIR.iterdir() if p.is_file() and p.suffix.lower() in EXTS])

for img_path in img_paths:
    print(f"--- Processing: {img_path.name} ---")
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        continue
    h, w = img_bgr.shape[:2]

    # --- YOLO 偵測 ---
    results = yolo_model.predict(source=img_bgr, conf=CONF_THRES, iou=IOU_THRES, imgsz=IMG_SIZE, device=GPU_ID, verbose=False)
    if not results or results[0].boxes is None or len(results[0].boxes) == 0:
        continue

    # 取信心度最高的 BBox
    boxes = results[0].boxes
    conf = boxes.conf.cpu().numpy()
    best_idx = int(conf.argmax())
    best_box = boxes.xyxy.cpu().numpy().astype(np.float32)[best_idx]
    
    # 確保座標在影像範圍內
    bx1, by1, bx2, by2 = map(int, best_box)
    bx1, bx2 = max(0, bx1), min(w, bx2)
    by1, by2 = max(0, by1), min(h, by2)
    box_w, box_h = bx2 - bx1, by2 - by1

    # ===========================
    # 輸出 0: 原圖畫上 YOLO BBox
    # ===========================
    vis_img = img_bgr.copy()
    cv2.rectangle(vis_img, (bx1, by1), (bx2, by2), (0, 255, 0), 4)
    cv2.imwrite(str(VIS_DIR / f"{img_path.stem}_bbox.jpg"), vis_img)

    # ===========================
    # 策略 3: Crop + Padding -> SAM
    # ===========================
    pad_x = int(box_w * PAD_RATIO)
    pad_y = int(box_h * PAD_RATIO)
    
    # 計算擴張後的座標，確保不超出原圖
    px1, px2 = max(0, bx1 - pad_x), min(w, bx2 + pad_x)
    py1, py2 = max(0, by1 - pad_y), min(h, by2 + pad_y)
    
    crop_padded = img_bgr[py1:py2, px1:px2]
    if crop_padded.size > 0:
        sam2.set_image(crop_padded)
        
        # 計算原始 BBox 在這張 Pad 小圖中的相對座標作為 Prompt
        rel_bx1, rel_by1 = bx1 - px1, by1 - py1
        rel_bx2, rel_by2 = bx2 - px1, by2 - py1
        
        masks_s3 = sam2.predict_with_boxes(np.array([[rel_bx1, rel_by1, rel_bx2, rel_by2]]))
        res_s3 = apply_mask_and_crop(crop_padded, masks_s3[0])
        if res_s3 is not None:
            cv2.imwrite(str(OUTPUT_DIR / f"{img_path.stem}_final.png"), res_s3)

print("\n[DONE] Processing complete.")

Visible CUDA devices: 1
[INFO] Loading YOLO model...
[INFO] Loading SAM2 model on cuda:2
[INFO] Models loaded.

--- Processing: 1J5A1090.JPG ---
--- Processing: 1J5A1091.JPG ---
--- Processing: 1J5A1956.JPG ---

[DONE] Processing complete.


In [2]:
import transformers as t
print(t.__version__)

5.2.0


## Sort

In [6]:
import os
import cv2
import torch
import shutil
import numpy as np
from pathlib import Path
from tqdm import tqdm
from transformers import AutoImageProcessor, AutoModel
from sklearn.decomposition import PCA

# ===========================
# 參數設定
# ===========================
DEVICE = "cuda:2" if torch.cuda.is_available() else "cpu"
# 輸入來源：剛剛 SAM2 產出的透明 PNG 資料夾
INPUT_DIR = Path("/home/nas2/Workspace/Aaron/phal_research/raw_images_0302/sam2_final_output")
# 輸出目標：重新命名並排序後存放的新資料夾
OUTPUT_DIR = Path("/home/nas2/Workspace/Aaron/phal_research/raw_images_0302/sorted_orchids")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ===========================
# 載入 DINOv2 模型 (提取特徵用)
# ===========================
print("[INFO] Loading DINOv3 model...")
# 使用 small 版本就足以提取極佳的全局特徵，且速度飛快
model_name = "facebook/dinov3-vitl16-pretrain-lvd1689m"
processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(DEVICE)
model.eval()

# ===========================
# 輔助函數：將 RGBA 轉換為純白背景 RGB
# ===========================
def rgba_to_white_bg_rgb(img_path):
    # 讀取包含 Alpha 通道的圖 (IMREAD_UNCHANGED)
    img_bgra = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)
    if img_bgra is None or img_bgra.shape[2] != 4:
        return None
    
    # 分離通道
    b, g, r, a = cv2.split(img_bgra)
    img_rgb = cv2.merge((r, g, b)) # 轉成 RGB
    
    # 建立一張純白背景 (255, 255, 255)
    white_bg = np.ones_like(img_rgb) * 255
    
    # 利用 Alpha 通道作為遮罩，將花朵貼上純白背景
    alpha_factor = a[:, :, np.newaxis] / 255.0
    blended = img_rgb * alpha_factor + white_bg * (1 - alpha_factor)
    
    return blended.astype(np.uint8)

# ===========================
# 步驟 1: 提取所有圖片的 DINO 特徵
# ===========================
print("[INFO] Extracting DINO features...")
img_paths = sorted([p for p in INPUT_DIR.glob("*.png")])
features = []
valid_paths = []

with torch.no_grad():
    for path in tqdm(img_paths):
        # 轉換為純白背景圖片
        img_rgb = rgba_to_white_bg_rgb(path)
        if img_rgb is None:
            continue
            
        # 前處理送入 DINO
        inputs = processor(images=img_rgb, return_tensors="pt").to(DEVICE)
        outputs = model(**inputs)
        
        # 提取 [CLS] token (shape: [1, hidden_size])，代表整張圖的全局特徵
        cls_feature = outputs.last_hidden_state[:, 0, :].cpu().numpy().flatten()
        
        features.append(cls_feature)
        valid_paths.append(path)

features = np.array(features)

# ===========================
# 步驟 2: PCA 降維至 1D 並排序
# ===========================
print("[INFO] Performing PCA to 1D...")
# 將高維度特徵壓成 1 維，找出變異量最大的一條軸
pca = PCA(n_components=1)
scores_1d = pca.fit_transform(features).flatten()

# 根據 1D 分數進行 argsort (由小到大排序)
sorted_indices = np.argsort(scores_1d)

# ===========================
# 步驟 3: 重新命名並複製到新資料夾
# ===========================
print("[INFO] Renaming and copying files...")
for rank, idx in enumerate(tqdm(sorted_indices)):
    original_path = valid_paths[idx]
    score = scores_1d[idx]
    
    # 建立新檔名：例如 001_Phal_score_-4.23.png
    # 保留分數可以讓你知道它們在光譜上的距離
    new_filename = f"{rank:03d}_score_{score:+.2f}_{original_path.name}"
    new_path = OUTPUT_DIR / new_filename
    
    shutil.copy(str(original_path), str(new_path))

print(f"\n[DONE] Sorted {len(valid_paths)} images saved to {OUTPUT_DIR}")

[INFO] Loading DINOv3 model...


Loading weights: 100%|██████████| 415/415 [00:01<00:00, 343.93it/s, Materializing param=norm.weight]                     


[INFO] Extracting DINO features...


100%|██████████| 192/192 [03:12<00:00,  1.00s/it]


[INFO] Performing PCA to 1D...
[INFO] Renaming and copying files...


100%|██████████| 192/192 [00:06<00:00, 30.55it/s]


[DONE] Sorted 192 images saved to /home/nas2/Workspace/Aaron/phal_research/raw_images_0302/sorted_orchids
